# LRGB Peptides-func Graph Classification with GNNVisualizer

This notebook trains a graph-level GAT model on the PyTorch Geometric `LRGBDataset(name="Peptides-func")` benchmark, then renders the trained model with `GNNVisualizer`.

`Peptides-func` is a chemistry graph classification dataset from the Long Range Graph Benchmark. PyG reports about 15,535 graphs, 150.94 nodes per graph, and 307.30 edges per graph, making it a useful AI4Science dataset for testing visualizations at slightly above the 100-node scale.

Source docs: [PyG LRGBDataset](https://pytorch-geometric.readthedocs.io/en/2.6.0/generated/torch_geometric.datasets.LRGBDataset.html) and [LRGB Peptides-func dataset card](https://huggingface.co/datasets/LRGB/peptides-functional/blob/main/README.md).

If imports fail in a fresh kernel, install the runtime packages first:

```bash
python3 -m pip install torch torch-geometric
```

Optional environment variables: `PEPTIDES_EPOCHS`, `PEPTIDES_MAX_TRAIN_GRAPHS`, `PEPTIDES_SCAN_GRAPHS`, `PEPTIDES_TARGET_NODES`, and `PEPTIDES_HIDDEN_CHANNELS`.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.datasets import LRGBDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

from gnn_exp import GNNVisualizer

In [ ]:
SEED = 7
torch.manual_seed(SEED)

EPOCHS = int(os.environ.get("PEPTIDES_EPOCHS", "5"))
MAX_TRAIN_GRAPHS = int(os.environ.get("PEPTIDES_MAX_TRAIN_GRAPHS", "256"))
SCAN_GRAPHS = int(os.environ.get("PEPTIDES_SCAN_GRAPHS", "1024"))
TARGET_NODES = int(os.environ.get("PEPTIDES_TARGET_NODES", "100"))
HIDDEN_CHANNELS = int(os.environ.get("PEPTIDES_HIDDEN_CHANNELS", "16"))
BATCH_SIZE = 32


def prepare_graph(data):
    data = data.clone()
    data.x = data.x.float()
    data.y = data.y.float().view(1, -1)
    return data


dataset = LRGBDataset(root=str(repo_root / "data" / "lrgb"), name="Peptides-func", split="train")
scan_count = min(SCAN_GRAPHS, len(dataset))
scan_graphs = [prepare_graph(dataset[index]) for index in range(scan_count)]
visual_index = min(range(scan_count), key=lambda index: abs(scan_graphs[index].num_nodes - TARGET_NODES))
visual_data = scan_graphs[visual_index]

generator = torch.Generator().manual_seed(SEED)
train_count = min(MAX_TRAIN_GRAPHS, len(dataset))
train_indices = torch.randperm(len(dataset), generator=generator)[:train_count].tolist()
train_graphs = [prepare_graph(dataset[int(index)]) for index in train_indices]
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)

query_pair = [0, min(visual_data.num_nodes - 1, max(1, visual_data.num_nodes // 2))]
num_features = visual_data.num_features
out_channels = int(visual_data.y.numel())

node_counts = torch.tensor([graph.num_nodes for graph in scan_graphs], dtype=torch.float)
edge_counts = torch.tensor([graph.edge_index.size(1) for graph in scan_graphs], dtype=torch.float)
display(Markdown(
    f"Peptides-func train split loaded with **{len(dataset):,} graphs**. "
    f"This notebook trains on **{len(train_graphs):,} sampled graphs** for a fast visual demo. "
    f"The first **{scan_count:,} graphs** average **{node_counts.mean():.1f} nodes** and **{edge_counts.mean():.1f} directed edges**. "
    f"The visualized graph is index `{visual_index}` with **{visual_data.num_nodes} nodes**, "
    f"**{visual_data.edge_index.size(1)} directed edges**, **{num_features} node features**, "
    f"and **{out_channels} binary function labels**."
))

In [ ]:
class PeptidesGAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        if hidden_channels % 2 != 0:
            raise ValueError("hidden_channels must be divisible by 2")
        heads = 2
        per_head_channels = hidden_channels // heads
        self.conv1 = GATConv(in_channels, per_head_channels, heads=heads, concat=True)
        self.act1 = nn.Tanh()
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1, concat=False)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)

In [ ]:
def target_from_batch(batch):
    return batch.y.float().view(batch.num_graphs, -1)


def masked_bce_loss(logits, target):
    mask = ~torch.isnan(target)
    return F.binary_cross_entropy_with_logits(logits[mask], target[mask])


def multilabel_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.x, batch.edge_index, batch.batch)
            target = target_from_batch(batch)
            mask = ~torch.isnan(target)
            pred = (torch.sigmoid(logits) >= 0.5).float()
            correct += int((pred[mask] == target[mask]).sum())
            total += int(mask.sum())
    return correct / max(total, 1)


def train_model(model, loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
    losses = []
    for _ in range(epochs):
        model.train()
        for batch in loader:
            optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index, batch.batch)
            loss = masked_bce_loss(logits, target_from_batch(batch))
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
    return {
        "final_loss": losses[-1],
        "sample_accuracy": multilabel_accuracy(model, loader),
    }


torch.manual_seed(SEED)
model = PeptidesGAT(num_features, HIDDEN_CHANNELS, out_channels)
metrics = train_model(model, train_loader)
model.eval()

display(Markdown(
    "| Metric | Value |\n"
    "|---|---:|\n"
    f"| Final training BCE | {metrics['final_loss']:.4f} |\n"
    f"| Sample label accuracy | {metrics['sample_accuracy']:.3f} |"
))

The next cell builds the widget. The displayed model keeps the raw encoded peptide atom features as the visible node feature matrix, then visualizes the learned GAT message-passing stack and graph-level readout.

In [ ]:
visualizer = GNNVisualizer()
visualizer.add_model(
    data=visual_data,
    model=model,
    subgraphSample=False,
    queries=[query_pair],
    mode="graph",
)

assert visualizer.modelInfo["conv1"]["type"] == "GATConv"
assert visualizer.modelInfo["conv1"].get("aggregation") == "attention"
assert len(visualizer.graphData["x"]) == visual_data.num_nodes
assert "graphAggregation" in visualizer.intmData
assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS

display(Markdown(
    "| Captured item | Value |\n"
    "|---|---:|\n"
    f"| First layer | `{visualizer.modelInfo['conv1']['type']}` |\n"
    f"| Aggregation | `{visualizer.modelInfo['conv1'].get('aggregation')}` |\n"
    f"| Graph pooling | `{visualizer.intmData['graphAggregation']['type']}` |\n"
    f"| Hidden width | {len(visualizer.intmData['act1'][0])} |\n"
    f"| Visualized nodes | {len(visualizer.graphData['x'])} |\n"
    f"| Query | `{visualizer.queries}` |"
))

In [ ]:
display(visualizer)